# 11 — Backtest de l'indicateur directionnel

> **Question :** L'indicateur est-il réellement utile ? Quand est-il fiable ?

| | |
|---|---|
| **Hypothèse** | DA(confidence élevée) ≫ DA(confidence faible) — le signal filtre correctement. |
| **Méthode** | `run_indicator_backtest()` — 9882 observations OOS, 4 horizons, 2015-2025. |
| **Rapport** | `docs/INDICATOR_BACKTEST_REPORT.md` |


In [1]:
import sys, warnings, os
sys.path.insert(0, '../../../src')
os.chdir('../../../')  # ROOT so relative paths like data/ artefacts/ work
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
ROOT_PATH = __import__('pathlib').Path('.')
from mais.indicator.backtest import run_indicator_backtest
import pandas as pd, numpy as np

# Charger le rapport déjà généré
report_path = ROOT_PATH / 'docs/INDICATOR_BACKTEST_REPORT.md'
if report_path.exists():
    print("Rapport disponible :", report_path)
    print(report_path.read_text()[:2000])
else:
    print("Rapport non disponible — lancer run_indicator_backtest()")


Rapport disponible : docs/INDICATOR_BACKTEST_REPORT.md
# Backtest de l'indicateur directionnel maïs

## Méthodologie

- Modèle : `ridge_factors`
- Observations totales (toutes horizons) : 9,882
- Évaluation walk-forward out-of-sample (même split que l'étude professionnelle)
- DA = Directional Accuracy = % de prédictions correctes (signal > 0.5 ↔ prix monte)
- Brier Score = erreur quadratique sur les probabilités (plus bas = meilleur)
- AUC = aire sous la courbe ROC (0.5 = aléatoire, 1.0 = parfait)
- Confidence = distance de P(up) par rapport à 0.5, normalisée

## Métriques globales par horizon

| Horizon | N | DA | Brier Score | AUC | CQR Coverage |
|---:|---:|---:|---:|---:|---:|
| J+5 | 2,475 | 0.562 | 0.2473 | 0.561 | 88.9% |
| J+10 | 2,473 | 0.583 | 0.2408 | 0.607 | 89.0% |
| J+20 | 2,469 | 0.616 | 0.2347 | 0.645 | 89.5% |
| J+30 | 2,465 | 0.625 | 0.2325 | 0.657 | 89.1% |

## Performance par niveau de confiance

> **Hypothèse clé** : le signal est plus fiable quand il est confiant.

## 1. Performance globale par horizon

In [2]:
# Charger les prédictions OOS directement
preds = pd.read_parquet('artefacts/professional_study/model_predictions.parquet')
model = 'ridge_factors'
preds_m = preds[preds['model']==model].copy()
preds_m['Date'] = pd.to_datetime(preds_m['Date'])

def sigmoid(x):
    return 1 / (1 + np.exp(-x / 0.036))

preds_m['prob_up'] = sigmoid(preds_m['y_pred'])
preds_m['pred_dir'] = (preds_m['prob_up'] > 0.5).astype(int)
preds_m['true_dir'] = (preds_m['y_true'] > 0).astype(int)
preds_m['correct'] = (preds_m['pred_dir'] == preds_m['true_dir']).astype(int)

da_by_h = preds_m.groupby('horizon')['correct'].mean()
print("Directional Accuracy par horizon (ridge_factors) :")
print(da_by_h.round(4).to_string())

ax = da_by_h.plot(kind='bar', figsize=(8, 4), color='steelblue')
ax.axhline(0.5, color='red', linestyle='--', label='Random (50%)')
ax.set_title('DA par horizon — ridge_factors OOS 2015-2025')
ax.set_ylabel('Directional Accuracy')
ax.set_xlabel('Horizon (jours)')
ax.legend()
plt.tight_layout()
plt.savefig('notebooks/corn_study/exports/11_da_by_horizon.png', dpi=100)
plt.show()


Directional Accuracy par horizon (ridge_factors) :
horizon
5     0.5616
10    0.5831
20    0.6160
30    0.6247


## 2. DA par niveau de confiance

In [3]:
from mais.research.uncertainty import summarize_cqr_results
cqr = pd.read_parquet('artefacts/professional_study/cqr_results.parquet')
cqr_m = cqr.merge(preds_m[['Date','horizon','prob_up','correct']], on=['Date','horizon'], how='inner')

# Proxy confidence = distance de prob_up à 0.5
cqr_m['confidence'] = (cqr_m['prob_up'] - 0.5).abs() * 2

# Tertiles de confidence
cqr_m['conf_bucket'] = pd.cut(
    cqr_m['confidence'],
    bins=[0, 0.3, 0.5, 1.01],
    labels=['low (<0.30)', 'medium (0.30-0.50)', 'high (>0.50)']
)
da_conf = cqr_m.groupby('conf_bucket', observed=True)['correct'].mean()
print("DA par niveau de confiance :")
print(da_conf.round(4).to_string())

ax = da_conf.plot(kind='bar', figsize=(8, 4), color=['salmon','khaki','mediumseagreen'])
ax.axhline(0.5, color='red', linestyle='--', label='Random')
ax.set_title('DA par niveau de confiance (proxy)')
ax.set_ylabel('Directional Accuracy')
ax.legend()
plt.tight_layout()
plt.savefig('notebooks/corn_study/exports/11_da_by_confidence.png', dpi=100)
plt.show()


DA par niveau de confiance :
conf_bucket
low (<0.30)           0.5653
medium (0.30-0.50)    0.5958
high (>0.50)          0.7608


## 3. DA par saison

In [4]:
preds_m['month'] = preds_m['Date'].dt.month
preds_m['season'] = preds_m['month'].map({
    12: 'winter', 1: 'winter', 2: 'winter',
    3: 'spring (semis)', 4: 'spring (semis)', 5: 'spring (semis)',
    6: 'summer', 7: 'summer', 8: 'summer',
    9: 'fall (récolte)', 10: 'fall (récolte)', 11: 'fall (récolte)',
})
da_season = preds_m.groupby('season')['correct'].mean().sort_values(ascending=False)
print("DA par saison :")
print(da_season.round(4).to_string())


DA par saison :
season
summer            0.6370
fall (récolte)    0.5937
spring (semis)    0.5831
winter            0.5723


## 4. DA par régime

In [5]:
regimes = pd.read_parquet('artefacts/professional_study/regime_timeseries.parquet')
regimes['Date'] = pd.to_datetime(regimes['Date'])
preds_with_regime = preds_m.merge(regimes[['Date','regime']], on='Date', how='left')
da_regime = preds_with_regime.groupby('regime')['correct'].agg(['mean','count'])
da_regime.columns = ['DA', 'N']
print("DA par régime :")
print(da_regime.round(4).to_string())


DA par régime :
            DA     N
regime              
bear    0.7602   492
bull    0.6173  5830
range   0.5393  3560


## 5. Interprétation

- **DA globale h30 : ~0.62** — significatif pour un marché financier.
- **DA confidence élevée >> DA confiance faible** → le confidence score filtre correctement.
- **Régime bear : DA ~0.76** mais sur peu d'observations (~2-5%) → résultat fragile.
- **Été : meilleure saison** (stress météo + WASDE dominants, marché plus prédictible).

## 6. Limites

- Confidence proxy utilisé (distance prob_up vs 0.5) — pas le vrai confidence score V1 qui inclut CQR width et stabilité.
- Un seul modèle (ridge_factors) — pas un stacking complet.
- Bear state : fragile (peu d'observations) → ETUDE-12 (DONE) a corrigé vers 2 états.

## 7. Décision

EXP-008 : L'indicateur est **utile sur les signaux confiants** (DA ~71% haute confiance vs ~56% basse confiance).
→ Recommander d'émettre UNCERTAIN quand confidence < 0.45 (seuil config/indicator.yaml).
→ Notebook 12 : synthèse finale des 6 questions de recherche.
